# Fine-tune TinyLlama as Noodle

**Before you start:**
1. Go to `Runtime → Change runtime type → T4 GPU` → Save
2. Upload `data/robot_chat.txt` from your project using the 📁 folder icon on the left
3. Run each cell top to bottom

Total time: ~20–30 minutes on Colab free GPU.

In [ ]:
# CELL 1 — Install dependencies
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
# CELL 2 — Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('ERROR: Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 3 — Load training data
# Make sure you uploaded data/robot_chat.txt first

SYSTEM_PROMPT = (
    "You are Noodle, a fun casual robot companion. "
    "Guy is your user. Guy is building a robot and likes coding. "
    "Be casual, short, and a bit cheeky."
)

def load_chat_data(path):
    """Convert Guy:/Noodle: lines into TinyLlama chat format."""
    examples = []
    lines = open(path, encoding='utf-8').read().strip().splitlines()

    i = 0
    while i < len(lines) - 1:
        if lines[i].startswith('Guy:') and lines[i+1].startswith('Noodle:'):
            user_msg = lines[i][len('Guy:'):].strip()
            bot_msg  = lines[i+1][len('Noodle:'):].strip()
            text = (
                f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
                f"<|user|>\n{user_msg}</s>\n"
                f"<|assistant|>\n{bot_msg}</s>"
            )
            examples.append(text)
            i += 2
        else:
            i += 1
    return examples

examples = load_chat_data('robot_chat.txt')
print(f'Loaded {len(examples)} training examples')
print('\nSample:')
print(examples[0])

In [ ]:
# CELL 4 — Load TinyLlama (downloads ~1.1 GB, takes ~2 min)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

# load in 4-bit — uses half the GPU memory, same quality
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading TinyLlama...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
print('Loaded. Parameters:', sum(p.numel() for p in model.parameters()) // 1_000_000, 'M')

In [ ]:
# CELL 5 — Apply LoRA
# LoRA only trains a tiny fraction of weights (fast, low memory)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# CELL 6 — Prepare dataset
from torch.utils.data import Dataset, DataLoader

MAX_LENGTH = 256

class ChatDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length):
        self.items = []
        for ex in examples:
            enc = tokenizer(
                ex,
                truncation=True,
                max_length=max_length,
                padding='max_length',
                return_tensors='pt',
            )
            self.items.append({
                'input_ids':      enc['input_ids'].squeeze(),
                'attention_mask': enc['attention_mask'].squeeze(),
                'labels':         enc['input_ids'].squeeze().clone(),
            })

    def __len__(self):  return len(self.items)
    def __getitem__(self, i): return self.items[i]

# repeat examples to give the model more to learn from
dataset = ChatDataset(examples * 8, tokenizer, MAX_LENGTH)
loader  = DataLoader(dataset, batch_size=4, shuffle=True)
print(f'Dataset size: {len(dataset)} examples')

In [ ]:
# CELL 7 — Train (~20-30 min on T4 GPU)
import time
from torch.optim import AdamW

EPOCHS = 5
optimizer = AdamW(model.parameters(), lr=2e-4)

model.train()
print(f'Training for {EPOCHS} epochs...')
print(f'{"Epoch":>6}  {"Step":>6}  {"Loss":>8}')
print('-' * 30)

for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].cuda()
        attention_mask = batch['attention_mask'].cuda()
        labels         = batch['labels'].cuda()

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    avg = total_loss / len(loader)
    print(f'{epoch+1:>6}         {avg:>8.4f}')

print('\nDone!')

In [ ]:
# CELL 8 — Save model
SAVE_PATH = 'noodle_model'

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Saved to {SAVE_PATH}/')

# Zip it for download
import shutil
shutil.make_archive('noodle_model', 'zip', SAVE_PATH)
print('Zipped as noodle_model.zip — download this file')

In [ ]:
# CELL 9 — Quick test (make sure it works before downloading)
from peft import PeftModel

model.eval()

def chat(user_input, temperature=0.8, max_new_tokens=80):
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}</s>\n"
        f"<|user|>\n{user_input}</s>\n"
        f"<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_k=40,
            pad_token_id=tokenizer.eos_token_id,
        )
    new = output[0, inputs['input_ids'].shape[1]:]
    reply = tokenizer.decode(new, skip_special_tokens=True)
    return reply.split('</s>')[0].split('<|')[0].strip()

print('Testing Noodle...')
tests = ['hey', 'what is your name', 'what can you do', 'tell me a joke']
for t in tests:
    print(f'Guy: {t}')
    print(f'Noodle: {chat(t)}\n')